# 01 - Text embeddings with `intfloat/multilingual-e5-large` for train/test similarity search

# Three Outliers:
> **Universitas Indonesia**
> - Eduardus Tjitrahardja
> - Ikhlasul Akmal Hanif
> - Rahmat Bryan Naufal

# Import and Installing Library

In [ ]:
DATA_DIR = "../data"

In [ ]:
!pip install -q CatBoost
!pip install -q datasets

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback
)

import tensorflow as tf
from catboost import CatBoostClassifier, Pool

from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, \
    balanced_accuracy_score
from datasets import Dataset, DatasetDict

In [ ]:
print(tf.config.list_physical_devices('GPU'))

[]


In [1]:
import pandas as pd
import re

# Loading and Cleaning Data

In [2]:
train_df = pd.read_csv(f'{DATA_DIR}/bdc_train.csv', sep=";")
test_df = pd.read_csv(f'{DATA_DIR}/bdc_test.csv', sep=";")
# train_df.head()

In [3]:
def remove_noise(text):
    text = str(text)
    # remove mentions
    text = re.sub(r'@[\S+]+', '', text)
    # remove hashtags
    text = re.sub(r'#\w+', '', text)
    # remove RT
    text = re.sub(r'RT[\s]+', '', text)
    # remove [RE ...]
    text = re.sub(r'\[RE\s+(\w+)\]', r'\1', text)
    # remove urls
    text = re.sub(r'https?:\/\/\S+', '', text)
    # remove emojis
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # fix &amp;
    text = re.sub(r'&amp;', 'dan', text)
    # fix w/
    text = re.sub(r'w/', 'with', text)
    # fix 3
    text = re.sub(r'(?<=\w)3(?=\w)', 'e', text)
    # remove any non alphanumeric characters
    text = re.sub(r'[^A-Za-z0-9 ]', ' ', text)
    # fix unspaced words
    new_text = []
    for word in text.split():
        # weird word
        if word in ["BEiqnH5xQr", "JN18A"]:
            continue
        # fix shorten names
        elif word == "JK":
            new_text.append("jusuf kalla")
        elif word.lower() == "jkw":
            new_text.append("jokowi")
        elif word.lower() in ["bowo", "ayahbowo"]:
            new_text.append("prabowo")
        elif word.lower() in ["ayahbowogbran", "pbowogbran", "masbowogbran"]:
            new_text.append("pabrowo gibran")
        elif word == "MasGBRAN":
            new_text.append("gibran")
        elif word == "SBY":
            new_text.append("susilo bambang yudhoyono")
        elif word == "AHY":
            new_text.append("agnes haryani")
        # keep as is
        elif re.fullmatch(r'[A-Z]+', word) or word.lower() in ["s1", "s2", "s3"]:
            new_text.append(word)
        else:
            # fix unspaced words
            new_text.append(re.sub(r'(?<=[a-z])(?=[A-Z])|(?<=[A-Za-z])(?=[0-9])|(?<=[0-9])(?=[A-Za-z])|(?<=[A-Z])(?=[A-Z][a-z])', ' ', word))
    text = ' '.join(new_text)
    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    # remove whitespace leading & trailing
    text = text.strip()
    return text

def normalize_word(text):
    map = {
        "sj": "saja",
        "yg": "yang",
        "dgn": "dengan",
        "dg": "dengan",
        "dpt": "dapat",
        "dptkan": "dapatkan",
        "hnya": "hanya",
        "skrng": "sekarang",
        "uu": "undang undang",
        "asn": "aparatur sipil negara",
        "ordal": "orang dalam",
        "mrk": "mereka",
        "rmh": "rumah",
        "kpk": "komisi pemberantasan korupsi",
        "blg": "bilang",
        "tuch": "tuh",
        "umkm": "usaha mikro kecil menengah",
        "jis": "jakarta international stadium",
        "jkt": "jakarta",
        "spt": "seperti",
        "tps": "tempat pemungutan suara",
        "alkes": "alat kesehatan",
        "bowo": "prabowo",
        "maap": "maaf",
        "ga" : "tidak",
        "nggak" : "tidak"
    }

    text = text.replace("UU ITE", "Undang Undang Informasi dan Transaksi Elektronik")
    text = text.replace("kitamasbowo dangbran", "kita pilih prabowo dan gibran")
    text = text.replace("PDI P", "PDIP")
    text = text.replace("PDI Perjuangan", "PDIP")

    party_names = [
        "pkb",
        "gerindra",
        "pdi",
        "pdip",
        "golkar",
        "nasdem",
        "gelora",
        "pks",
        "pkn",
        "hanura",
        "pan",
        "pbb",
        "demokrat",
        "psi",
        "perindo",
        "ppp",
        "ummat",
        "pna",
        "gabthat",
        "pda",
        "sira"
    ]

    stop_words = [
        "dekade08",
        "tempodotco",
        "md"
    ]

    new_text = []
    for word in text.split():
        if word.lower() in map:
            new_text.append(map[word.lower()])
        elif word.lower() in party_names:
            new_text.append("Partai Politik")
        elif word.lower() in stop_words:
            continue
        else:
            new_text.append(word)

    text = ' '.join(new_text)
    return text.lower()

def drop_duplicated(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text', 'label'], keep='first', inplace=True)
    print("After dropping literal duplicated rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

def drop_duplicated_test(df):
    print("Df size:", df.shape)
    df.drop(df[df['text'].str.len() == 0].index, inplace=True)
    print("After dropping empty text rows, df size: ", df.shape)
    df.drop_duplicates(subset=['text'], keep=False, inplace=True)
    print("After dropping text duplicated rows, df size: ", df.shape)
    return df

In [4]:
# train_df['text'] = train_df['text'].apply(replace_hashtags)
train_df['text'] = train_df['text'].apply(remove_noise)
train_df['text'] = train_df['text'].apply(normalize_word)
# train_df = drop_duplicated(train_df)
train_df.head()

                                                text             label
0  kunjungan prabowo ini untuk meresmikan dan men...  Sumber Daya Alam
1  anies dapat tepuk tangan meriah saat jadi rekt...           Politik
2  emng bener sih pendukung 01 ada yang goblok be...         Demografi
3  sewaktu anies bersikap kritis ke kinerja pak p...           Politik
4  anies baswedan harap aparatur sipil negara ter...           Politik

In [5]:
test_df.rename(columns={'Text': 'text'}, inplace=True)

In [6]:
test_df['text'] = test_df['text'].apply(remove_noise)
test_df['text'] = test_df['text'].apply(normalize_word)
test_df.head()

    IDText                                               text
0  TXT0001  lu mau org 2 pro demokrasi di negara ini bisa ...
1  TXT0002  prabowo ditanya soal hutang luar negeri dia me...
2  TXT0003  kiki daliyo ganjar pranowo itulah beliau sosok...
3  TXT0004  prabowo gibran yang bisa melakukan itu semua d...
4  TXT0005  lah justru yang gak nyambung junjungan elu aom...

# Get The Embedding

In [7]:
!pip install -q sentence_transformers

In [8]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(f'intfloat/multilingual-e5-large')

def e5_embed(query: str):
    embeddings = embedder.encode(sentences=[query], convert_to_tensor=False, normalize_embeddings=True)
    return embeddings

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [9]:
train_df['text_embed'] = train_df['text'].apply(e5_embed)
test_df['text_embed'] = test_df['text'].apply(e5_embed)

In [10]:
def tempo(text_raw):
    return text_raw[0]
train_df['text_embed'] = train_df['text_embed'].apply(tempo)
test_df['text_embed'] = test_df['text_embed'].apply(tempo)

In [11]:
train_df.to_pickle(f"{DATA_DIR}/with_embed_full.pkl")

In [ ]:
test_df.head()

    IDText                                               text
0  TXT0001  lu mau org 2 pro demokrasi di negara ini bisa ...
1  TXT0002  prabowo ditanya soal hutang luar negeri dia me...
2  TXT0003  kiki daliyo ganjar pranowo itulah beliau sosok...
3  TXT0004  prabowo gibran yang bisa melakukan itu semua d...
4  TXT0005  lah justru yang gak nyambung junjungan elu aom...

In [ ]:
test_df['text_embed'] = test_df['text'].apply(e5_embed)

In [ ]:
def tempo(text_raw):
    return text_raw[0]
test_df['text_embed'] = test_df['text_embed'].apply(tempo)

In [12]:
test_df.to_pickle(f"{DATA_DIR}/with_embed_test_full.pkl")